**Part 2 · LangChain** · step 15c of 25
[← 15b Streaming](15b_streaming.ipynb) · [🏠 Roadmap](../README.md) · [16 First graph →](../03_LangGraph/16_first_graph.ipynb)

# 15c · RAG: Retrieval-Augmented Generation

An LLM only knows its training data. It doesn't know **your** documents (company policies, product docs,
tickets) or anything newer than its cutoff. Pasting every document into the prompt doesn't scale:
the context window is limited, it costs a lot, and quality drops with very long prompts.

**RAG** = find the few relevant chunks first, then give **only those** to the model.

```
INDEXING (once, offline)                          QUERY (every question)
───────────────────────                           ──────────────────────
documents                                         question
  │ 1. load      (PDF, web, DB…)                    │ embed
  │ 2. split     into chunks                        ▼
  │ 3. embed     each chunk → vector       ──►  vector store: find top-k similar chunks
  ▼ 4. store     in a vector store                  │
vector store                                        ▼
                                                  prompt = question + chunks → LLM → grounded answer
```

You'll build both styles:
- **2-step RAG** (a chain): always retrieve, then answer. Fast and predictable.
- **Agentic RAG**: retrieval is a **tool**, and the agent decides when and what to search.

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

## 1. Load documents

Everything in LangChain RAG is a `Document`: `page_content` (text) + `metadata` (source, page, category…).
Loaders turn files into `Document`s:

```python
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader, DirectoryLoader
docs = PyPDFLoader("handbook.pdf").load()         # one Document per page, metadata has "page"
docs = DirectoryLoader("docs/", glob="**/*.md", loader_cls=TextLoader).load()
```

To keep this notebook self-contained we create the documents directly. The company is **fictional**,
so the model *cannot* know these answers from training. Any correct answer must come from retrieval.

In [ ]:
from langchain_core.documents import Document

raw_docs = [
    Document(
        page_content="""Nimbus Labs Leave Policy.
Full-time employees get 24 days of paid annual leave per calendar year. Leave accrues monthly at 2 days per month.
Up to 8 unused days can be carried over to the next year; anything above 8 days expires on 31 March.
Sick leave is separate: 10 days per year, and a doctor's note is required for absences longer than 3 consecutive days.
Parental leave is 26 weeks fully paid for the primary caregiver and 6 weeks for the secondary caregiver.""",
        metadata={"source": "hr/leave_policy.md", "category": "hr"},
    ),
    Document(
        page_content="""Nimbus Labs Remote Work Policy.
Employees may work remotely up to 3 days per week. Tuesdays and Thursdays are mandatory office days for team rituals.
Fully remote arrangements need written approval from a director. The company pays a one-time home office
allowance of 500 USD and a monthly internet allowance of 40 USD. Working from another country is allowed
for at most 30 days per year and must be registered with HR at least two weeks in advance.""",
        metadata={"source": "hr/remote_work.md", "category": "hr"},
    ),
    Document(
        page_content="""Nimbus Labs IT Security Rules.
All laptops must use full-disk encryption and lock automatically after 5 minutes of inactivity.
Passwords must be at least 14 characters, and multi-factor authentication through the Nimbus Authenticator app is mandatory.
Lost or stolen devices must be reported to security@nimbuslabs.example within 1 hour.
Personal USB storage devices are blocked on company laptops. VPN is required on public Wi-Fi.""",
        metadata={"source": "it/security.md", "category": "it"},
    ),
    Document(
        page_content="""Nimbus Labs Expense Policy.
Meals during business travel are reimbursed up to 60 USD per day. Economy class is required for flights under
6 hours; business class is allowed above 6 hours with manager approval. Expenses must be submitted in the
ExpenseHub tool within 30 days with a receipt. Expenses above 1000 USD need VP approval.""",
        metadata={"source": "finance/expenses.md", "category": "finance"},
    ),
    Document(
        page_content="""NimbusDB Product FAQ.
NimbusDB is Nimbus Labs' managed vector database. The free tier includes 1 million vectors and 2 GB storage.
The Pro plan costs 99 USD per month and includes 50 million vectors, daily backups and a 99.9% uptime SLA.
NimbusDB supports cosine, dot-product and euclidean distance. The maximum vector dimension is 4096.
Data is stored in the Frankfurt and Virginia regions; customers choose the region at creation time.""",
        metadata={"source": "product/nimbusdb_faq.md", "category": "product"},
    ),
]
len(raw_docs)

## 2. Split into chunks

Why split? A retriever returns whole chunks. If chunks are too **big**, you send irrelevant text (costly,
noisy). If they're too **small**, a chunk loses its context ("it costs 99 USD": what does?).

`RecursiveCharacterTextSplitter` is the default choice. It tries to split on paragraphs, then lines, then
sentences, then words, so chunks stay meaningful.

| Setting | Meaning | Typical values |
|---|---|---|
| `chunk_size` | max characters per chunk | 500–1500 for prose |
| `chunk_overlap` | characters repeated between neighbouring chunks, so a fact cut in half still appears whole | 10–20% of `chunk_size` |

Our documents are tiny, so we use small chunks to actually see the splitting.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50, add_start_index=True)
chunks = splitter.split_documents(raw_docs)

print(f"{len(raw_docs)} documents → {len(chunks)} chunks\n")
for c in chunks[:3]:
    print(c.metadata, "\n", c.page_content, "\n" + "-" * 60)

Metadata from the parent document is copied to every chunk, so we always know the **source**.
That's what lets the answer cite where it came from.

## 3. Embeddings: text → vector

An embedding model turns text into a list of numbers so that **similar meaning → nearby vectors**.
Similarity is usually measured with **cosine similarity** (1 = same direction, ~0 = unrelated).

In [ ]:
import numpy as np
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

sentences = [
    "How many vacation days do I get?",
    "Employees receive 24 days of paid annual leave.",
    "The free tier includes 1 million vectors.",
]
vectors = np.array(embeddings.embed_documents(sentences))
print("vector length:", vectors.shape[1])


def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


print(f"question vs leave sentence  : {cosine(vectors[0], vectors[1]):.3f}")
print(f"question vs product sentence: {cosine(vectors[0], vectors[2]):.3f}")

"Vacation days" and "annual leave" share **no keywords**, yet they score much closer than the unrelated
sentence. That's the point of semantic search over keyword search.

## 4. Vector store: store and search the vectors

`InMemoryVectorStore` needs no setup, which makes it good for learning. Every vector store has the same
interface, so switching to FAISS, Chroma or pgvector in production is a one-line change.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)  # embeds every chunk once

for doc, score in vector_store.similarity_search_with_score("How many vacation days do I get?", k=3):
    print(f"{score:.3f}  {doc.metadata['source']:<22} {doc.page_content[:70]}...")

In [ ]:
# Same thing with FAISS (installed in this project): fast, and can be saved to disk and reloaded.
# FAISS's LangChain wrapper lives in langchain-community, which is being sunset, so you'll see a
# DeprecationWarning. It still works. New projects should prefer standalone packages (langchain-chroma, langchain-postgres…).
import tempfile

from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(chunks, embeddings)

with tempfile.TemporaryDirectory() as folder:
    faiss_store.save_local(folder)  # index once...
    reloaded = FAISS.load_local(folder, embeddings, allow_dangerous_deserialization=True)  # ...reuse later
    print(reloaded.similarity_search("Can I use a USB stick?", k=1)[0].page_content)

| Vector store | When to pick it |
|---|---|
| `InMemoryVectorStore` | learning, tests |
| FAISS | local, very fast, saved as files; no server |
| Chroma | local or server, simple, metadata filtering |
| pgvector (Postgres) | you already run Postgres; vectors next to your business data |
| Pinecone / Weaviate / Qdrant | managed, large scale |

## 5. Retrievers: the standard "query in → documents out" interface

`vector_store.as_retriever()` wraps a store as a **Runnable**, so it can be used in LCEL chains.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
for d in retriever.invoke("What is the internet allowance?"):
    print(d.metadata["source"], "→", d.page_content[:90], "...")

**MMR (Maximal Marginal Relevance)**: plain top-k can return near-duplicate chunks (especially with
overlap). MMR fetches `fetch_k` candidates and then picks `k` that are relevant **and** different from
each other.

**Metadata filtering**: restrict the search to a subset (one department, one customer, one date range).
In multi-tenant apps this is how you make sure user A never retrieves user B's documents.

In [ ]:
mmr_retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10})
print("MMR:", [d.metadata["source"] for d in mmr_retriever.invoke("company rules for employees")])

# InMemoryVectorStore takes a Python function as filter; Chroma/pgvector take a dict like {"category": "hr"}
hr_only = vector_store.as_retriever(
    search_kwargs={"k": 3, "filter": lambda doc: doc.metadata["category"] == "hr"}
)
print("HR only:", [d.metadata["source"] for d in hr_only.invoke("company rules for employees")])

## 6. 2-step RAG as an LCEL chain

Retrieve → stuff the chunks into the prompt → answer. Two new LCEL pieces:
- A **dict** in a chain = `RunnableParallel`: each key is computed from the same input.
- `RunnablePassthrough()` passes the input through unchanged (here: the question).

The prompt tells the model to answer **only** from the context and to say so when the answer isn't there.
This is your main defence against hallucination.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4o-mini", temperature=0)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using ONLY the context below. "
     "If the answer is not in the context, say only \"I don't know based on the documents.\" "
     "Otherwise end with the source file(s) you used, in brackets.\n\nContext:\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

print(rag_chain.invoke("How many unused leave days can I carry over, and when do extra days expire?"))

In [ ]:
# Not in the documents → the model should refuse instead of inventing an answer
print(rag_chain.invoke("What is the dress code at Nimbus Labs?"))

## 7. Agentic RAG: retrieval as a tool

The 2-step chain **always** retrieves exactly once with the user's exact words. An agent can:
- skip retrieval for small talk ("hi"),
- **rewrite** a vague question into a better search query,
- search **several times** for multi-part questions, then combine the results.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool


@tool
def search_handbook(query: str) -> str:
    """Search the Nimbus Labs internal documents (HR, IT security, finance/expenses, NimbusDB product).
    Use a short, specific search query. Call it once per sub-question."""
    return format_docs(retriever.invoke(query))


rag_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_handbook],
    system_prompt=(
        "You are the Nimbus Labs internal assistant. For any question about the company, its policies or "
        "products, use search_handbook. Answer only from the search results and cite sources in brackets. "
        "If the results don't contain the answer, say you don't know."
    ),
)

result = rag_agent.invoke({"messages": [HumanMessage(
    "I'm flying 8 hours to a client and might lose my laptop there. What class can I fly, "
    "and what do I do if the laptop is stolen?"
)]})

for m in result["messages"]:
    if getattr(m, "tool_calls", None):
        print("🔎 searches:", [c["args"]["query"] for c in m.tool_calls])
print("\n" + result["messages"][-1].content)

In [ ]:
# Small talk: the agent answers without searching (no wasted retrieval)
r = rag_agent.invoke({"messages": [HumanMessage("Hi! Who are you?")]})
print("tool calls:", sum(len(getattr(m, "tool_calls", []) or []) for m in r["messages"]))
print(r["messages"][-1].content)

| | 2-step RAG (chain) | Agentic RAG (tool) |
|---|---|---|
| Retrieval | always, exactly once | when the model decides, 0..n times |
| Query | the user's raw words | rewritten by the model |
| Latency / cost | low, predictable | higher (extra model calls) |
| Best for | FAQ bots, single-topic Q&A | multi-part questions, mixed chat + lookup, many sources |

Add a checkpointer for follow-ups ("and for the secondary caregiver?"). Add a metadata filter built from
**runtime context** (15a) so each user only searches their own documents.

## 8. Making RAG better: what interviews ask about

| Problem | Technique |
|---|---|
| Exact terms (IDs, error codes, names) missed by embeddings | **Hybrid search**: BM25 keyword + vector, merge results (`EnsembleRetriever`) |
| Top-k contains loosely related chunks | **Re-ranking**: fetch 20, re-score with a cross-encoder (e.g. Cohere Rerank), keep 3–5 |
| Vague or conversational question ("what about the other one?") | **Query rewriting** with chat history; **multi-query** (generate several phrasings) |
| Chunks lose context | tune `chunk_size`/overlap; parent-document retrieval (search small chunks, return the larger parent) |
| Wrong tenant's data retrieved | **metadata filters** from trusted context, never from the model |
| Is it actually good? | **Evaluate**: retrieval (context precision / recall) and generation (faithfulness, answer relevance) with LangSmith or RAGAS on a test set |
| Stale data | re-index on change; store `updated_at` in metadata |

**When *not* to use RAG:** the data fits easily in the prompt (just include it), you need exact numbers from
structured data (use a SQL tool, see project 21), or you need to change the model's *style* rather than its knowledge (fine-tuning or prompting).

## Revision notes
- RAG = **retrieve** relevant chunks, then **generate** an answer grounded in them. Fixes unknown/private/fresh data.
- Pipeline: load → split (`RecursiveCharacterTextSplitter`, `chunk_size`, `chunk_overlap`) → embed → store → retrieve.
- Embeddings map meaning to vectors; cosine similarity finds related text without shared keywords.
- `vector_store.as_retriever(search_type="similarity"|"mmr", search_kwargs={"k":…, "filter":…})`.
- LCEL RAG: `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | model | parser`.
- Agentic RAG: wrap the retriever in a `@tool`; the agent decides when/how often to search.
- Prompt "answer only from context, else say you don't know" + cite sources → fewer hallucinations.
- Production upgrades: hybrid search, re-ranking, query rewriting, metadata filters, evaluation.

---
✅ **Done with 15c RAG.** Next: [16 First graph →](../03_LangGraph/16_first_graph.ipynb)
[🏠 Roadmap](../README.md)